In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

d = np.load(f"{SAVE_DIR}/CHBMIT_crosspatient.npy", allow_pickle=True).item()
Xtr, ytr = d["train"]["X"], d["train"]["y"]
Xte, yte = d["test"]["X"], d["test"]["y"]

FS = 256
CH = ["FP1-F7","F7-T7","T7-P7","P7-O1","FP1-F3","F3-C3","C3-P3","P3-O1","FP2-F4",
      "F4-C4","C4-P4","P4-O2","FP2-F8","F8-T8","T8-P8","P8-O2","FZ-CZ","CZ-PZ"]

print("train", Xtr.shape, "test", Xte.shape)
print("train seizure", int(ytr.sum()), "of", len(ytr))
print("test seizure", int(yte.sum()), "of", len(yte))
print("amplitude range", float(Xtr.min()), float(Xtr.max()))

In [ ]:
seiz = Xtr[ytr == 1][0]
norm = Xtr[ytr == 0][0]
t = np.arange(seiz.shape[1]) / FS

fig, axes = plt.subplots(1, 2, figsize=(14, 9), sharey=True)
for ax, sig, title in zip(axes, [norm, seiz], ["Non-seizure", "Seizure"]):
    offset = 0
    step = np.abs(Xtr).max() * 0.5
    for i in range(sig.shape[0]):
        ax.plot(t, sig[i] + offset, linewidth=0.5, color="black")
        offset -= step
    ax.set_yticks(-step * np.arange(len(CH)))
    ax.set_yticklabels(CH, fontsize=8)
    ax.set_xlabel("Time (s)")
    ax.set_title(title)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_windows.png", dpi=150)
plt.show()

In [ ]:
def bandpower(X, lo, hi):
    freqs = np.fft.rfftfreq(X.shape[-1], 1/FS)
    psd = np.abs(np.fft.rfft(X, axis=-1))**2
    m = (freqs >= lo) & (freqs < hi)
    return psd[..., m].sum(-1)

bands = {"delta":(0.5,4), "theta":(4,8), "alpha":(8,13), "beta":(13,30), "gamma":(30,50)}
fig, ax = plt.subplots(figsize=(8,5))
w = 0.35
for i, (name,(lo,hi)) in enumerate(bands.items()):
    bp = bandpower(Xtr, lo, hi).mean(1)
    ax.bar(i - w/2, np.log10(bp[ytr==0].mean()), w, color="grey", label="Non-seizure" if i==0 else "")
    ax.bar(i + w/2, np.log10(bp[ytr==1].mean()), w, color="black", label="Seizure" if i==0 else "")
ax.set_xticks(range(len(bands)))
ax.set_xticklabels(bands.keys())
ax.set_ylabel("log10 mean band power")
ax.legend()
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_bands.png", dpi=150)
plt.show()

In [ ]:
def corrmat(X):
    mats = []
    for x in X:
        sd = x.std(axis=1)
        if np.any(sd == 0):
            continue
        mats.append(np.corrcoef(x))
    return np.nanmean(np.array(mats), axis=0), len(mats)

c0, n0 = corrmat(Xtr[ytr==0])
c1, n1 = corrmat(Xtr[ytr==1])
print(f"used {n0} non-seizure and {n1} seizure windows (flat-channel windows dropped)")

def offdiag_mean(m):
    iu = np.triu_indices_from(m, k=1)
    return np.abs(m[iu]).mean()

print(f"mean absolute off-diagonal correlation")
print(f"  non-seizure {offdiag_mean(c0):.3f}")
print(f"  seizure     {offdiag_mean(c1):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, m, title in zip(axes, [c0, c1, c1-c0], ["Non-seizure", "Seizure", "Difference"]):
    diff = title == "Difference"
    im = ax.imshow(m, cmap="bwr" if diff else "gray",
                   vmin=-0.3 if diff else -1, vmax=0.3 if diff else 1)
    ax.set_title(title)
    ax.set_xticks(range(18)); ax.set_xticklabels(CH, rotation=90, fontsize=6)
    ax.set_yticks(range(18)); ax.set_yticklabels(CH, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/fig_corr.png", dpi=150)
plt.show()

In [ ]:
print("NaNs in raw train X:", np.isnan(Xtr).sum())
print("NaNs in raw test X:", np.isnan(Xte).sum())

def corr_features(X, y):
    iu = np.triu_indices(X.shape[1], k=1)
    feats, labs, dropped = [], [], 0
    for x, lab in zip(X, y):
        with np.errstate(invalid="ignore", divide="ignore"):
            c = np.corrcoef(x)[iu]
        if np.all(np.isfinite(c)):
            feats.append(c); labs.append(lab)
        else:
            dropped += 1
    return np.array(feats), np.array(labs), dropped

Ftr, ltr, dtr = corr_features(Xtr, ytr)
Fte, lte, dte = corr_features(Xte, yte)
print(f"dropped {dtr} train, {dte} test windows with degenerate channels")
print(f"kept {len(Ftr)} train, {len(Fte)} test")
print(f"kept test seizure fraction: {lte.mean():.3f}")

clf = LogisticRegression(max_iter=2000)
clf.fit(Ftr, ltr)
acc = accuracy_score(lte, clf.predict(Fte))
print(f"seizure detection from correlation structure ALONE: {acc:.4f}")
print(f"majority-class baseline: {1 - lte.mean():.4f}")

In [ ]:
def bp_features(X):
    feats = []
    for lo, hi in [(0.5,4),(4,8),(8,13),(13,30),(30,50)]:
        feats.append(np.log10(bandpower(X, lo, hi) + 1e-12))
    return np.concatenate(feats, axis=1)

def corr_features_aligned(X):
    iu = np.triu_indices(X.shape[1], k=1)
    feats, keep = [], []
    for i, x in enumerate(X):
        with np.errstate(invalid="ignore", divide="ignore"):
            c = np.corrcoef(x)[iu]
        if np.all(np.isfinite(c)):
            feats.append(c); keep.append(i)
    return np.array(feats), np.array(keep)

Ctr, ktr = corr_features_aligned(Xtr)
Cte, kte = corr_features_aligned(Xte)

Btr = bp_features(Xtr)[ktr]
Bte = bp_features(Xte)[kte]

ytr_k = ytr[ktr]
yte_k = yte[kte]

combos = {
    "band power only": (Btr, Bte),
    "correlation only": (Ctr, Cte),
    "band power + correlation": (np.hstack([Btr, Ctr]), np.hstack([Bte, Cte])),
}

print(f"majority-class baseline: {1 - yte_k.mean():.4f}\n")
for name, (tr, te) in combos.items():
    clf = LogisticRegression(max_iter=2000)
    clf.fit(tr, ytr_k)
    acc = accuracy_score(yte_k, clf.predict(te))
    print(f"{name}: {acc:.4f}  (dim={tr.shape[1]})")